# kernelport: freeze references on a free Colab GPU

Validating a reference runs the **original CUDA kernel**, so it needs an NVIDIA GPU. If you work on a Mac
(or any machine without one), let your agent write the references and input specs there, commit
`.kernelport/`, and run this notebook once to validate and freeze them on Colab's free T4.

Then pull the frozen `.kernelport/` back and let your agent port and verify on your own machine:
`verify_port` only needs the frozen reference, not the GPU it was frozen on.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv

In [ ]:
# Colab already ships CUDA torch and CuPy; this adds kernelport and the NVRTC wheel.
!pip -q install "kernelport[cuda]"
!kernelport doctor

## Your repo

Set `REPO` to your repository (with the candidate references in `.kernelport/` committed).
For a private repo, use a URL with a read token, or upload a zip of the repo and unzip it to `/content/repo`.

In [ ]:
REPO = "https://github.com/<you>/<your-repo>"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
import os
if not os.path.isdir("/content/repo"):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/repo
%cd /content/repo
!kernelport status

In [ ]:
# Runs each original CUDA kernel against its candidate reference on every case and freezes the
# ones that PASS. A FAIL means the REFERENCE is wrong: fix it on your machine and run this again.
!kernelport validate --all
!kernelport status

In [ ]:
# Download the workspace and unzip it over your repo's .kernelport/, then commit.
!zip -qr /content/kernelport_frozen.zip .kernelport
from google.colab import files
files.download("/content/kernelport_frozen.zip")